# Continuous Binning — Parameter Sensitivity Analysis

Sweeps the bias node parameter grid defined in `config.py`, fits a `ContinuousBinningModel`
for each combo, and runs `generate_parameter_sensitivity_report()` to identify stable
parameter regions and produce interactive visualizations.

**Workflow:**
1. Edit `config.py` (or override inline below) to set tickers, date range, bias_spec
2. Run all cells (Kernel → Restart & Run All)
3. Interact with the Plotly figures in Section 5

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import pandas as pd
from datetime import datetime
from dataclasses import replace

from feature_research.continuous_binning.config import load_config
from feature_research.continuous_binning.data_loader import (
    expand_bias_specs,
    load_features_for_combo,
    populate_cache_if_needed,
    param_combo_label,
)
from feature_selection.base_models.continuous_binning import ContinuousBinningModel
from metrics.performance import SortinoRatio
from eda.parameter_analysis import generate_parameter_sensitivity_report
from utils.enums import Ticker, TimeFrame

print("Imports OK")

## 1. Configuration

Edit `config.py` for persistent changes, or use the override cell below for one-off runs.
`dataclasses.replace()` is used because `ResearchConfig` is a frozen dataclass.

In [ ]:
# --- Load defaults from config.py ---
config = load_config()

# ============================================================
# OVERRIDE CELL — edit here to customise without touching config.py
# ============================================================

# config = replace(config, tickers=[Ticker.ES])
# config = replace(config, start=datetime(2015, 1, 1), end=datetime(2024, 12, 31))
# config = replace(config,
#     bias_spec={
#         "module_name": "rsi",
#         "timeframes": [TimeFrame.D],
#         "params": {"lookback": [5, 10, 14, 20, 30]},
#     }
# )

# --- Param sensitivity knobs ---
STABILITY_THRESHOLD = 0.8
TOP_K = 3
METRIC_COL = "sortino"
N_BINS = config.binning_params.bin_counts[0]

# ============================================================

# Derive varying param names (list-valued params only)
varying_params = [
    k for k, v in config.bias_spec["params"].items()
    if isinstance(v, list) and len(v) > 1
]
if not varying_params:
    # Fallback: use all params if none have multiple values
    varying_params = list(config.bias_spec["params"].keys())

print(f"Module   : {config.bias_spec['module_name']}")
print(f"Tickers  : {[t.name for t in config.tickers]}")
print(f"Period   : {config.start.date()} -> {config.end.date()}")
print(f"Varying  : {varying_params}")
print(f"N_BINS   : {N_BINS}  |  METRIC: {METRIC_COL}  |  THRESHOLD: {STABILITY_THRESHOLD}")

## 2. Feature Extraction

Loads feature + target data for every param combo. Uses the cache if `config.use_cache=True`.
Set `config.populate_cache=True` to auto-populate missing cache entries.

In [ ]:
populate_cache_if_needed(config)

expanded_specs = expand_bias_specs(config.bias_spec)
print(f"Parameter grid: {len(expanded_specs)} combinations\n")

combo_data = {}
for spec in expanded_specs:
    result = load_features_for_combo(spec, config)
    if result is not None:
        feature, target, feature_col = result
        combo_data[param_combo_label(spec["params"])] = {
            "feature": feature,
            "target": target,
            "params": spec["params"],
        }
    else:
        print(f"  SKIP: {spec['params']}")

print(f"\nLoaded {len(combo_data)}/{len(expanded_specs)} combos successfully")

## 3. Metric Grid

Fits `ContinuousBinningModel` for each combo and computes the Sortino ratio on selected
returns. Builds `results_df` with `param{K}_value` columns required by
`generate_parameter_sensitivity_report()`.

In [ ]:
metric = SortinoRatio(annualization_factor=252)
rows = []

for spec in expanded_specs:
    label = param_combo_label(spec["params"])
    if label not in combo_data:
        continue

    data = combo_data[label]
    feature = data["feature"]
    target = data["target"]

    try:
        model = ContinuousBinningModel(n_bins=N_BINS)
        model.fit(feature, target)
        signals = model.predict(feature, strategy="long")
        signals = np.asarray(signals).astype(bool).flatten()
        selected_returns = target.values[signals]

        if len(selected_returns) < 5:
            selected_returns = target.values

        metric_value = metric.compute(selected_returns)

        row = {
            f"param{k+1}_value": spec["params"][key]
            for k, key in enumerate(varying_params)
        }
        row[METRIC_COL] = metric_value
        rows.append(row)

    except Exception as e:
        print(f"  ERROR [{label}]: {e}")

results_df = pd.DataFrame(rows)
print(f"Grid results: {len(results_df)} rows")
results_df

## 4. Parameter Sensitivity Report

Runs: neighbor smoothing → stable region detection → recommendations.

In [ ]:
report = generate_parameter_sensitivity_report(
    results_df=results_df,
    param_names=varying_params,
    metric_col=METRIC_COL,
    stability_threshold=STABILITY_THRESHOLD,
    top_k=TOP_K,
)

print(f"Module              : {config.bias_spec['module_name']}")
print(f"Metric              : {report.metric_name}")
print(f"Grid size           : {report.n_parameter_combinations}")
print(f"Stable regions      : {report.n_stable_regions}")
print(f"Mean stability ratio: {report.mean_stability_ratio:.3f}")
print(f"% stable combos     : {report.pct_stable_combinations:.1%}")
print(f"\nTop-{TOP_K} recommendations:")
for i, combo in enumerate(report.top_k_combinations, 1):
    params_str = ", ".join(
        f"{name}={val}" for name, val in zip(varying_params, combo)
    )
    print(f"  {i}. {params_str}")

## 5. Interactive Visualizations

The appropriate plot is selected automatically based on grid dimensionality:
- 1D grid → raw + smoothed line with shaded stable regions
- 2D grid → heatmap with stability contour overlay
- 3D+ grid → heatmap slices with dropdown + slider controls

In [ ]:
if report.plot_1d is not None:
    report.plot_1d.show()

if report.plot_2d is not None:
    report.plot_2d.show()

if report.plot_3d is not None:
    report.plot_3d.show()

## 6. Optional Save

Uncomment cells below to save plots as interactive HTML or grid results as parquet.

In [ ]:
# --- OPTIONAL: Save outputs (uncomment to enable) ---

# from pathlib import Path
# save_dir = config.reports_dir / "param_sensitivity"
# save_dir.mkdir(parents=True, exist_ok=True)

# if report.plot_1d is not None:
#     report.plot_1d.write_html(save_dir / f"param_sensitivity_1d_{config.bias_spec['module_name']}.html")
#     print(f"Saved 1D plot -> {save_dir}")

# if report.plot_2d is not None:
#     report.plot_2d.write_html(save_dir / f"param_sensitivity_2d_{config.bias_spec['module_name']}.html")
#     print(f"Saved 2D plot -> {save_dir}")

# if report.plot_3d is not None:
#     report.plot_3d.write_html(save_dir / f"param_sensitivity_3d_{config.bias_spec['module_name']}.html")
#     print(f"Saved 3D plot -> {save_dir}")

# report.grid_results.to_parquet(save_dir / "param_sensitivity_grid.parquet")
# print(f"Saved grid results -> {save_dir / 'param_sensitivity_grid.parquet'}")